# 1-2 자동 미분

본 노트북은 본문 1-2 절의 코드 예제(1-42 ~ 1-48)를 모은 것이다. 파이토치 연산 그래프와 자동 미분(Autograd)의 동작 방식을 살펴본 뒤, 자동 미분 결과(기울기)를 사용해 카페 비용 최적화 예제의 파라미터(재료별 단위가격)를 직접 갱신해 본다.

## 연산 그래프와 자동 미분

연산 그래프는 텐서끼리의 연산이 진행되는 순서를 기록한 그래프이다. `requires_grad=True` 로 만든 텐서가 포함된 연산은 이 그래프에 등록되고, 출력 텐서의 `backward()` 메서드를 호출하면 그래프를 거슬러 올라가며 자동 미분(역전파)이 수행된다. 자동 미분의 결과는 각 텐서의 `grad` 속성에 저장된다.

본문 [코드 1-42]는 입력 텐서 `x`, 파라미터 텐서 `a`, `b` 에 대해 `y = sum(a * x + b)` 의 순전파 -> 역전파 흐름을 보여 준다. 본문은 `x`, `a`, `b` 를 미리 만들어 두었다고 가정하므로, 노트북에서는 본문 코드 직전에 같은 형태의 텐서를 만드는 준비 셀을 추가한다.

In [1]:
# 참고 - [코드 1-27] 실행을 위한 텐서 준비와 시드 고정
# x 는 입력 텐서이므로 requires_grad=False (기본값) 인 그대로 둔다.
# a, b 는 파라미터에 해당하므로 requires_grad=True 로 만든다.
import random

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

x = torch.randn((2, ))
a = torch.randn((2, ), requires_grad=True)
b = torch.randn((2, ), requires_grad=True)

print(f'x = {x}')
print(f'a = {a}')
print(f'b = {b}')

x = tensor([0.3367, 0.1288])
a = tensor([0.2345, 0.2303], requires_grad=True)
b = tensor([-1.1229, -0.1863], requires_grad=True)


In [ ]:
# 코드 1-27 - 순전파와 역전파의 실행

# x, a, b: randn()으로 미리 만들어 둔 (2, ) 형태의 텐서
# torch.sum() 또는 텐서의 sum() 메서드: 요소의 값을 더하는 집계 함수/메서드
y = torch.sum(a * x + b)        # 순전파
y.backward()                    # 역전파

# 자동 미분으로 구한 기울기는 grad 속성을 통해 확인 가능
print(a.grad, b.grad)           # a, b와 같은 형태의 텐서

tensor([0.3367, 0.1288]) tensor([1., 1.])


`y = sum(a * x + b)` 를 `a` 로 편미분하면 `x` 가, `b` 로 편미분하면 `1` 이 된다. 따라서 `a.grad` 는 입력 `x` 와 같은 값, `b.grad` 는 모든 요소가 1인 텐서가 된다. (시드를 다르게 두면 `a.grad` 의 구체적 수치는 달라지지만, `a.grad == x` 와 `b.grad == [1., 1.]` 의 관계는 그대로 유지된다.)

## 자동 미분 대상 텐서 만들기

미분의 대상이 되는 텐서는 다음 두 가지 방식 중 하나로 표시한다.

1. 텐서를 생성할 때 `requires_grad=True` 키워드 인자 지정
2. 만들어진 텐서의 `requires_grad` 속성을 `True` 로 변경 (`tensor.requires_grad = True` 또는 제자리 연산 메서드 `tensor.requires_grad_()`)

In [ ]:
# 코드 1-28 - 자동 미분의 대상 텐서로 지정

# requires_grad=True로 텐서 생성
a = torch.tensor([1., 2.], requires_grad=True)

# requires_grad를 지정하지 않으면 requires_grad 속성의 기본값은 False가 됨
b = torch.randn((2, ))
b.requires_grad = True                              # 속성 변경으로도 지정 가능
# 또는 b.requires_grad_() 메서드를 사용해도 됨

print(f'a.requires_grad = {a.requires_grad}')
print(f'b.requires_grad = {b.requires_grad}')

a.requires_grad = True
b.requires_grad = True


정수형 텐서는 불연속 값을 가지므로 미분이 정의되지 않는다. 따라서 `requires_grad=True` 로 만들 수 없고, 시도하면 `RuntimeError` 가 발생한다.

In [4]:
# 코드 1-28(계속) - 정수형 텐서는 requires_grad=True 지정이 불가

c = torch.tensor([1, 2], requires_grad=True)    # RuntimeError 예외 발생

RuntimeError: Only Tensors of floating point and complex dtype can require gradients

## 자동 미분 사용 예 - 카페 비용 모델

카페라테와 코코아 두 음료의 1일 총 비용을 계산하는 모델을 만들어, 자동 미분으로 재료별 단위가격의 영향력을 구하고, 그 영향력을 사용해 단위가격을 반복적으로 인하해 비용을 줄여 본다.

먼저 자동 미분 없이 단위가격의 1일 총 비용을 계산해 본다.

In [ ]:
# 코드 1-29 - 음료의 단위가격과 1일 총 비용 계산

# 카페라테와 코코아 1잔에 필요한 커피, 우유, 초콜릿, 전기, 물의 사용량
ingredients = [[10, 10, 0, 8, 2], [0, 15, 10, 5, 1]]

# 카페라테와 코코아의 판매량
sales_1day = [20, 10]

# 재료별 단위가격
unit_costs = [100, 100, 100, 100, 100]

ingredients = torch.tensor(ingredients)         # 재료 텐서
sales_1day = torch.tensor(sales_1day)           # 판매량 텐서
unit_costs = torch.tensor(unit_costs)           # 재료별 단위가격 텐서

# 음료별 생산비용을 계산해 반환하는 함수
def get_prices(unit_costs, ingredients):
    return (unit_costs * ingredients).sum(dim=1)

# 전체 비용을 계산해 반환하는 함수
def get_cost(prices, sales):
    return sum(prices * sales)

prices = get_prices(unit_costs, ingredients)    # 음료별 생산비용 텐서
cost_1day = get_cost(prices, sales_1day)        # 1일 총 비용 텐서
print(f'카페라테와 코코아 1잔의 가격: {prices}')
# item() 메서드: 스칼라 텐서나 크기 1 텐서를 정수 또는 실수로 바꾸는 메서드
print(f'1일 총 비용: {cost_1day.item()}')

카페라테와 코코아 1잔의 가격: tensor([3000, 3100])
1일 총 비용: 91000


In [6]:
# 참고 - item() 메서드는 스칼라 텐서 또는 크기 1의 텐서에만 사용할 수 있다.

torch.tensor([1, 2]).item()         # RuntimeError 예외 발생

RuntimeError: a Tensor with 2 elements cannot be converted to Scalar

### 참고 - 차원을 기준으로 동작하는 집계 메서드

본문 보조 설명에서 다룬 `sum()`, `max()` 메서드의 두 가지 동작 방식을 함께 확인해 본다. `dim` 인자 없이 호출하면 모든 요소를 하나의 스칼라로 집계하고, `dim` 인자를 주면 그 차원만 따라 집계한 텐서를 만든다.

In [ ]:
# 차원을 기준으로 동작하는 집계 메서드

square_tensor = torch.tensor([[1, 4], [2, 3]])
print(square_tensor.sum())          # 네 수의 합
print(square_tensor.sum(dim=0))     # 1+2, 4+3
print(square_tensor.sum(dim=1))     # 1+4, 2+3

print(square_tensor.max())          # 가장 큰 수
# dim=1: 1과 2, 4와 3 중 최댓값(values)과 그 인덱스(indices)
print(square_tensor.max(dim=0)) 
# dim=0: 1과 4, 2와 3 중 최댓값(values)과 그 인덱스(indices)
print(square_tensor.max(dim=1)) 

tensor(10)
tensor([3, 7])
tensor([5, 5])
tensor(4)
torch.return_types.max(
values=tensor([2, 4]),
indices=tensor([1, 0]))
torch.return_types.max(
values=tensor([4, 3]),
indices=tensor([1, 1]))


### 자동 미분으로 unit_costs 의 기울기 구하기

이제 `unit_costs` 텐서를 자동 미분의 대상으로 지정한 뒤, 1일 총 비용(`cost_1day`)에 대한 기울기를 계산한다. 기울기는 각 재료의 단위가격이 1원 변할 때 1일 총 비용이 얼마나 변하는지를 의미한다.

In [ ]:
# 코드 1-30 - 자동 미분으로 unit_costs 텐서의 기울기를 계산하는 절차

# 자동 미분 대상 텐서로 지정
# (정수형 텐서라면 지정하기 전 실수형 텐서로 변환해야 함)
unit_costs = unit_costs.to(torch.float)
unit_costs.requires_grad = True

# 순전파: cost_1day(목적 텐서) 계산 과정의 연산 그래프 생성
prices = get_prices(unit_costs, ingredients)
cost_1day = get_cost(prices, sales_1day)

# 역전파: cost_1day의 연산 그래프를 사용해 대상 텐서의 기울기 계산
cost_1day.backward()

# 자동 미분으로 구한 기울기: grad 속성 변수에 저장
print(f'각 재료 단위가격의 영향력(기울기): {unit_costs.grad}')

각 재료 단위가격의 영향력(기울기) - tensor([200., 350., 100., 210.,  50.])


기울기 텐서의 각 요소는 차례로 커피, 우유, 초콜릿, 전기, 물의 단위가격이 1일 총 비용에 미치는 영향력에 해당한다. 우유(두 번째 요소)의 단위가격을 1원 낮추면 비용이 350원 줄어들고, 물(마지막 요소)을 1원 낮추면 50원 줄어드는 식이다. 영향력이 가장 큰 재료는 우유임을 확인할 수 있다.

### 기울기로 파라미터 최적화하기

영향력의 1% 씩 단위가격을 낮추는 과정을 10회 반복해 1일 총 비용을 줄여 본다. 매 반복 단계는 다음과 같다.

1. 이전 기울기를 초기화 (`unit_costs.grad = None`)
2. 순전파 - 현재 단위가격으 1일 총 비용 계산 (연산 그래프 생성)
3. 역전파 - `total_cost.backward()` 로 단위가격의 기울기 계산
4. 최적화 - `torch.no_grad()` 컨텍스트 안에서 복합 할당 연산으로 단위가격 갱신

In [ ]:
# 코드 1-31 - 10회 반복 최적화로 비용 줄이기

iteration = 10              # 최적화 반복 횟수
discount_rate = 0.01        # 영향력을 반영해 단위가격을 인하하는 비율

print(f'초기 비용: {get_cost(get_prices(unit_costs, ingredients), sales_1day)}')

for i in range(iteration):
    # 기울기 초기화: 초기화하지 않으면 이전 미분 결과에 누적됨
    unit_costs.grad = None
    # 순전파
    total_cost = get_cost(get_prices(unit_costs, ingredients), sales_1day)
    # 역전파
    total_cost.backward()
    # torch.no_grad()로 생성한 컨텍스트 속의 연산은 연산 그래프를 만들지 않음
    with torch.no_grad():
        # 최적화: 재료별로 영향력의 일정비율만큼 단위가격을 낮춤
        unit_costs -= discount_rate * unit_costs.grad
        # unit_costs = unit_costs - discount_rate * unit_costs.grad
        #     -> RuntimeError (할당 연산은 자동 미분 대상에서 분리시킴)

        # 최적화 과정 출력
        print(f'{i + 1}번째 최적화 이후:')
        print(f'    단위가격: {unit_costs.tolist()}')
        print(f'    비용: {get_cost(get_prices(unit_costs, ingredients), sales_1day)}')

초기 비용: 91000.0
1번째 최적화 이후 :
    단위가격: [98.0, 96.5, 99.0, 97.9000015258789, 99.5]
    비용: 88809.0
2번째 최적화 이후 :
    단위가격: [96.0, 93.0, 98.0, 95.80000305175781, 99.0]
    비용: 86618.0
3번째 최적화 이후 :
    단위가격: [94.0, 89.5, 97.0, 93.70000457763672, 98.5]
    비용: 84427.0
4번째 최적화 이후 :
    단위가격: [92.0, 86.0, 96.0, 91.60000610351562, 98.0]
    비용: 82236.0
5번째 최적화 이후 :
    단위가격: [90.0, 82.5, 95.0, 89.50000762939453, 97.5]
    비용: 80045.0
6번째 최적화 이후 :
    단위가격: [88.0, 79.0, 94.0, 87.40000915527344, 97.0]
    비용: 77854.0
7번째 최적화 이후 :
    단위가격: [86.0, 75.5, 93.0, 85.30001068115234, 96.5]
    비용: 75663.0
8번째 최적화 이후 :
    단위가격: [84.0, 72.0, 92.0, 83.20001220703125, 96.0]
    비용: 73472.0
9번째 최적화 이후 :
    단위가격: [82.0, 68.5, 91.0, 81.10001373291016, 95.5]
    비용: 71281.0
10번째 최적화 이후 :
    단위가격: [80.0, 65.0, 90.0, 79.00001525878906, 95.0]
    비용: 69090.0


반복할수록 1일 총 비용이 줄어든다. 영향력이 가장 큰 우유의 단위가격이 가장 많이 줄었고, 영향력이 가장 작은 물의 단위가격은 가장 적게 줄었다.

주의할 두 가지:

1. **복합 할당 연산(`-=`) 사용**: `unit_costs = unit_costs - ...` 와 같은 일반 할당은 결과로 새 텐서를 만들어 `requires_grad=False` 가 된다. `unit_costs -= ...` 는 텐서의 값만 바꾸므로 `requires_grad` 속성이 유지된다.
2. **`torch.no_grad()` 컨텍스트**: 최적화 자체나 중간 결과 출력은 연산 그래프에 기록할 필요가 없다. `with torch.no_grad():` 컨텍스트 안에서 수행하면 그래프가 만들어지지 않아 메모리를 절약할 수 있다.

다음 절(1-3)에서는 이번 절에서 다룬 자동 미분과 경사하강법을 활용해 본격적인 회귀 분석 딥러닝 모델을 만들고 학습한다.